In [32]:
import pandas as pd 
import numpy as np 
from pathlib import Path 
print("Libraries loaded successfully")

Libraries loaded successfully


In [33]:
file_path = "../data/cleaned/paimana_all_months_combined.csv" 
df = pd.read_csv(file_path) 
print("Dataset loaded successfully") 
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Dataset loaded successfully
Rows: 7594
Columns: 18


In [34]:
pd.set_option("display.max_columns", None) 
df.head()

,MINISTRY,SECTOR,SL.NO,PROJECT NAME,AGENCY,PROJECT CODE(S),STATE,DATE OF APPROVAL,START DATE,ORIGINAL/TARGET DOC,REVISED DOC,ORIGINAL COST (RS. CRORE),REVISED COST (RS. CRORE),CUMULATIVE EXPENDITURE (RS. CRORE),PHYSICAL PROGRESS (%),SOURCE PAGE,REPORT MONTH,SOURCE FILE
0,DEPARTMENT OF TELECOMMUNICATIONS,TELECOMMUNICATION,1714.0,"AMENDED BHARATNET PROGRAM - KTK, GOA, PY",DEPARTMENT OF TELECOMMUNICATIONS,108841 | - | -,PAN INDIA,Aug-23,Jun-25,Jun-28,-,3742.0,3742.0,435.31,0.0815,149.0,JULY,Project_Monitoring_July_2026.xlsx
1,DEPARTMENT OF TELECOMMUNICATIONS,TELECOMMUNICATION,1800.0,"AMENDED BHARATNET PROGRAM - KTK, GOA, PY",DEPARTMENT OF TELECOMMUNICATIONS,108841 | - | -,PAN INDIA,Aug-23,Jun-25,Jun-28,-,3742.0,3742.0,435.31,0.0000,157.0,JUNE,Project_Monitoring_June_2026.xlsx
2,DEPARTMENT OF TELECOMMUNICATIONS,TELECOMMUNICATION,1715.0,AMENDED BHARATNET PROGRAM - HR,DEPARTMENT OF TELECOMMUNICATIONS,124199 | - | -,PAN INDIA,Aug-23,Mar-26,Mar-29,-,837.0,837.0,83.68,0.0000,149.0,JULY,Project_Monitoring_July_2026.xlsx
3,DEPARTMENT OF TELECOMMUNICATIONS,TELECOMMUNICATION,1716.0,AMENDED BHARATNET PROGRAM - PB,DEPARTMENT OF TELECOMMUNICATIONS,171218 | - | -,PAN INDIA,Aug-23,Feb-25,Feb-28,-,1245.0,1245.0,0.00,0.2658,149.0,JULY,Project_Monitoring_July_2026.xlsx
4,DEPARTMENT OF TELECOMMUNICATIONS,TELECOMMUNICATION,1717.0,"AMENDED BHARATNET PROGRAM - MP, DD AND DNH",DEPARTMENT OF TELECOMMUNICATIONS,175860 | - | -,PAN INDIA,Jun-23,May-25,May-28,-,4943.0,4943.0,562.13,0.0840,149.0,JULY,Project_Monitoring_July_2026.xlsx


In [35]:
print("All columns:\n") 
for i, column in enumerate(df.columns, 1):    
    print(i, "→", column)

All columns:

1 → MINISTRY
2 → SECTOR
3 → SL.NO
4 → PROJECT NAME
5 → AGENCY
6 → PROJECT CODE(S)
7 → STATE
8 → DATE OF APPROVAL
9 → START DATE
10 → ORIGINAL/TARGET DOC
11 → REVISED DOC
12 → ORIGINAL COST (RS. CRORE)
13 → REVISED COST (RS. CRORE)
14 → CUMULATIVE EXPENDITURE (RS. CRORE)
15 → PHYSICAL PROGRESS (%)
16 → SOURCE PAGE
17 → REPORT MONTH
18 → SOURCE FILE


In [36]:
df.columns = (    
    df.columns    
    .str.strip()    
    .str.upper()    
    .str.replace(" ", "_")    
    .str.replace("/", "_", regex=False)    
    .str.replace("(", "", regex=False)    
    .str.replace(")", "", regex=False)    
    .str.replace("%", "PCT", regex=False)    
    .str.replace(".", "", regex=False) 
)

In [37]:
missing = pd.DataFrame({    
    "Missing_Count": df.isna().sum(),    
    "Missing_Percentage": (        
        df.isna().mean() * 100 )
        .round(2) }) 
missing = missing.sort_values(    "Missing_Percentage",    ascending=False ) 
missing

,Missing_Count,Missing_Percentage
DATE_OF_APPROVAL,68,0.90
START_DATE,15,0.20
REVISED_DOC,15,0.20
SECTOR,4,0.05
ORIGINAL_TARGET_DOC,4,0.05
SLNO,4,0.05
PROJECT_CODES,4,0.05
AGENCY,4,0.05
STATE,4,0.05
PROJECT_NAME,4,0.05


In [38]:
duplicate_rows = df.duplicated().sum() 
print("Exact duplicate rows:", duplicate_rows)

Exact duplicate rows: 0


In [39]:
[col for col in df.columns if "PROJECT_CODE" in col]

['PROJECT_CODES']

In [40]:
PROJECT_ID = "PROJECT_CODES" 
print("Unique project codes:") 
print(df[PROJECT_ID].nunique())

Unique project codes:
3580


In [41]:
print(df["REPORT_MONTH"].value_counts() )

REPORT_MONTH
MAY      1987
APRIL    1981
JUNE     1847
JULY     1775
Name: count, dtype: int64


In [42]:
print(df.columns.tolist())

['MINISTRY', 'SECTOR', 'SLNO', 'PROJECT_NAME', 'AGENCY', 'PROJECT_CODES', 'STATE', 'DATE_OF_APPROVAL', 'START_DATE', 'ORIGINAL_TARGET_DOC', 'REVISED_DOC', 'ORIGINAL_COST_RS_CRORE', 'REVISED_COST_RS_CRORE', 'CUMULATIVE_EXPENDITURE_RS_CRORE', 'PHYSICAL_PROGRESS_PCT', 'SOURCE_PAGE', 'REPORT_MONTH', 'SOURCE_FILE']


In [43]:
date_columns = [    
    col for col in df.columns    
    if "DATE" in col or "DOC" in col ] 
print(date_columns)

['DATE_OF_APPROVAL', 'START_DATE', 'ORIGINAL_TARGET_DOC', 'REVISED_DOC']


In [45]:
for col in date_columns:    
    df[col] = pd.to_datetime(        
        df[col],
         errors="coerce"    
    ) 
print("Date conversion completed")

Date conversion completed


In [46]:
numeric_keywords = [    "COST",    "EXPENDITURE",    "PROGRESS" ] 
numeric_columns = [    
    col for col in df.columns    
    if any(word in col for word in numeric_keywords) ] 
print(numeric_columns)

['ORIGINAL_COST_RS_CRORE', 'REVISED_COST_RS_CRORE', 'CUMULATIVE_EXPENDITURE_RS_CRORE', 'PHYSICAL_PROGRESS_PCT']


In [47]:
for col in numeric_columns:    
    df[col] = (        
        df[col]        
        .astype(str)        
        .str.replace(",", "", regex=False)        
        .str.replace("₹", "", regex=False)        
        .str.strip()    )        
        
    df[col] = pd.to_numeric(
         df[col],        
         errors="coerce"    ) 
         
print("Numerical conversion completed")

Numerical conversion completed


In [48]:
df[numeric_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
ORIGINAL_COST_RS_CRORE,7594.0,3780.417685,82515.547734,102.4,355.9925,795.18000,1527.0000,3712662.01
REVISED_COST_RS_CRORE,7594.0,4291.038019,93818.004807,0.1,358.3000,796.97000,1661.2500,4278402.37
CUMULATIVE_EXPENDITURE_RS_CRORE,7594.0,2196.616750,48157.095093,0.0,54.2625,249.53500,728.7025,2196663.55
PHYSICAL_PROGRESS_PCT,7590.0,0.595984,0.345616,0.0,0.2800,0.68215,0.9230,1.00


In [49]:
progress_col = "PHYSICAL_PROGRESS_PCT"
print("Minimum:", df[progress_col].min()) 
print("Maximum:", df[progress_col].max())
print("Median:", df[progress_col].median())

Minimum: 0.0
Maximum: 1.0
Median: 0.68215


In [50]:
# part B 
df["PHYSICAL_PROGRESS_PERCENT"] = (df[progress_col] * 100)
df[    [        progress_col,        "PHYSICAL_PROGRESS_PERCENT"    ] ].head(20)


,PHYSICAL_PROGRESS_PCT,PHYSICAL_PROGRESS_PERCENT
0,0.0815,8.15
1,0.0000,0.00
2,0.0000,0.00
3,0.2658,26.58
4,0.0840,8.40
5,0.1513,15.13
6,0.0765,7.65
7,0.0000,0.00
8,0.0000,0.00
9,0.0179,1.79


In [51]:
# PART C - STANDARDIZE REPORT MONTH

df["REPORT_MONTH"] = (    df["REPORT_MONTH"]    
                      .astype(str)    
                      .str.strip()    
                      .str.upper() ) 
month_order = [    "APRIL",    "MAY",    "JUNE",    "JULY" ] 
df["REPORT_MONTH"] = pd.Categorical(    df["REPORT_MONTH"],    categories=month_order,    ordered=True ) 
print("Report month standardized successfully!") 
print(df["REPORT_MONTH"].value_counts())

Report month standardized successfully!
REPORT_MONTH
MAY      1987
APRIL    1981
JUNE     1847
JULY     1775
Name: count, dtype: int64


In [52]:
# PART C2 - CREATE CLEAN PROJECT ID 
df["PROJECT_ID"] = (    
    df["PROJECT_CODES"]
    .astype(str)    
    .str.split("|")    
    .str[0]    
    .str.strip() ) 

df["PROJECT_ID"] = (    
    pd.to_numeric(        
        df["PROJECT_ID"],        
        errors="coerce"    )    
        .astype("Int64")    
        .astype("string") ) 

print("Project ID created successfully!") 
print(df["PROJECT_ID"].head(10)) 
print("Unique Project IDs:", df["PROJECT_ID"].nunique())

Project ID created successfully!
0    108841
1    108841
2    124199
3    171218
4    175860
5    210914
6    222253
7    222253
8    240484
9    245986
Name: PROJECT_ID, dtype: string
Unique Project IDs: 2073


In [53]:
# CHECK PROJECT ID
print(    df[        [            "PROJECT_ID",            "PROJECT_CODES",            "REPORT_MONTH",            "SOURCE_FILE"        ]    ].head(20) )

   PROJECT_ID              PROJECT_CODES REPORT_MONTH  \
0      108841             108841 | - | -         JULY   
1      108841             108841 | - | -         JUNE   
2      124199             124199 | - | -         JULY   
3      171218             171218 | - | -         JULY   
4      175860             175860 | - | -         JULY   
5      210914             210914 | - | -         JULY   
6      222253             222253 | - | -         JULY   
7      222253             222253 | - | -         JUNE   
8      240484             240484 | - | -         JULY   
9      245986             245986 | - | -         JULY   
10     298178             298178 | - | -         JULY   
11     317725             317725 | - | -         JULY   
12     364934             364934 | - | -         JULY   
13     400005  400005 | N18000337 | 4690        APRIL   
14     400006             400006 | - | -         JULY   
15     400006             400006 | - | -         JUNE   
16     400006          400006 |

In [54]:
# PART d 

df = df.sort_values(    
    ["PROJECT_ID", "REPORT_MONTH"] ).reset_index(drop=True) 
print("Project history sorted successfully!") 
df[    
    ["PROJECT_ID",        
     "REPORT_MONTH"    ] 
].head(20)

Project history sorted successfully!


,PROJECT_ID,REPORT_MONTH
0,108841,JUNE
1,108841,JULY
2,124199,JULY
3,171218,JULY
4,175860,JULY
5,210914,JULY
6,222253,JUNE
7,222253,JULY
8,240484,JULY
9,245986,JULY


In [55]:
# PART E - PROJECT MONTH HISTORY 
project_month_check = (    
    df.groupby("PROJECT_ID")["REPORT_MONTH"]      
    .nunique()      
    .value_counts()      
    .sort_index() ) 
print("Number of months each project appears in:") 
print(project_month_check)

Number of months each project appears in:
REPORT_MONTH
1      68
2     175
3     151
4    1679
Name: count, dtype: int64


In [56]:
projects_all_4 = (    
    df.groupby("PROJECT_ID")["REPORT_MONTH"]      
    .nunique() ) 
projects_all_4 = projects_all_4[    projects_all_4 == 4 ]
print(    "Projects appearing in all 4 months:",    
      len(projects_all_4) )

Projects appearing in all 4 months: 1679


In [57]:
# PART F - DUPLICATE PROJECT/MONTH CHECK 
duplicate_project_month = df[    
    df.duplicated(        
        subset=["PROJECT_ID", "REPORT_MONTH"],        
        keep=False    ) ].sort_values(    ["PROJECT_ID", "REPORT_MONTH"] ) 
print(    "Duplicate project-month records:",    len(duplicate_project_month) ) 
duplicate_project_month.head(20)

Duplicate project-month records: 4


,MINISTRY,SECTOR,SLNO,PROJECT_NAME,AGENCY,PROJECT_CODES,STATE,DATE_OF_APPROVAL,START_DATE,ORIGINAL_TARGET_DOC,REVISED_DOC,ORIGINAL_COST_RS_CRORE,REVISED_COST_RS_CRORE,CUMULATIVE_EXPENDITURE_RS_CRORE,PHYSICAL_PROGRESS_PCT,SOURCE_PAGE,REPORT_MONTH,SOURCE_FILE,PHYSICAL_PROGRESS_PERCENT,PROJECT_ID
7590,TOTAL,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT,NaT,NaT,3712662.01,4278402.37,2036107.49,NaN,NaN,NaN,Project_Monitoring_April_2026.xlsx,NaN,<NA>
7591,TOTAL,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT,NaT,NaT,3370138.22,3710641.55,1926099.57,NaN,NaN,NaN,Project_Monitoring_July_2026.xlsx,NaN,<NA>
7592,TOTAL,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT,NaT,NaT,3561721.07,4054473.30,2196663.55,NaN,NaN,NaN,Project_Monitoring_June_2026.xlsx,NaN,<NA>
7593,TOTAL,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaT,NaT,NaT,3709724.65,4249554.14,2181683.19,NaN,NaN,NaN,Project_Monitoring_May_2026.xlsx,NaN,<NA>


In [61]:
# PART G - SAVE FINAL CLEANED DATA 
output_path = "../data/cleaned/paimana_cleaned1.csv" 
df.to_csv(    output_path,    index=False ) 
print("================================") 
print("CLEANED DATASET SAVED") 
print("================================") 
print("File:", output_path) 
print("Rows:", len(df)) 
print("Columns:", len(df.columns))

CLEANED DATASET SAVED
File: ../data/cleaned/paimana_cleaned1.csv
Rows: 7594
Columns: 20
